In [1]:
import os
import re
import warnings
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

try:
    from numpy.exceptions import RankWarning
except ImportError:
    from numpy import RankWarning

warnings.simplefilter('ignore', RankWarning)
warnings.filterwarnings('ignore')


In [2]:
DATASET_PATH    = "../data/dataset.csv"
FILLED_PATH     = "../data/filled_dataset.csv"
SUBMISSION_PATH = "../submissions/submission.csv"

df = pd.read_csv(DATASET_PATH)

feature_cols = [c for c in df.columns if c not in ['datetime', 'underlying_price']]
ce_cols = sorted([c for c in feature_cols if c.endswith('CE')])
pe_cols = sorted([c for c in feature_cols if c.endswith('PE')])

def parse_strike(col: str) -> int:
    """Extract 5-digit strike price from contract name e.g. NIFTY27JAN2625200CE -> 25200."""
    m = re.search(r'\d{5}', col)
    return int(m.group()) if m else 0

ce_strikes = np.array([parse_strike(c) for c in ce_cols], dtype=float)
pe_strikes = np.array([parse_strike(c) for c in pe_cols], dtype=float)

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"CE strikes: {ce_strikes.min():.0f} – {ce_strikes.max():.0f}  ({len(ce_cols)} contracts)")
print(f"PE strikes: {pe_strikes.min():.0f} – {pe_strikes.max():.0f}  ({len(pe_cols)} contracts)")
print(f"Total missing values: {df[feature_cols].isna().sum().sum()}")
print(f"Missing rate: {df[feature_cols].isna().sum().sum()/(df.shape[0]*len(feature_cols))*100:.1f}%")


Dataset: 975 rows x 30 columns
CE strikes: 26252 – 26265  (14 contracts)
PE strikes: 26238 – 26251  (14 contracts)
Total missing values: 5460
Missing rate: 20.0%


In [3]:
# --- Tuned hyperparameters ---
N_NEIGHBORS  = 5      # nearest observed strikes for local quadratic fit
CONV_THRESH  = 1e-8   # minimum leading coefficient to accept a convex fit
SPIKE_THRESH = 1.5    # max observed IV above which we skip quadratic (expiry spikes)
W_INTERIOR   = 0.80   # quadratic blend weight for interior gaps
W_EDGE       = 0.55   # quadratic blend weight for edge/extrapolation gaps
IV_FLOOR     = 0.001
IV_CAP       = 8.0

def _local_quad_estimate(obs_strikes: np.ndarray, obs_ivs: np.ndarray,
                          target: float) -> float:
    """
    Fits a 2nd-degree polynomial to the N nearest observed (strike, IV) pairs.
    Returns the interpolated value only if the curvature is positive (convex smile).
    A negative leading coefficient means the curve is concave — a noise artifact
    that would produce unrealistic values — so we discard it and return NaN.
    """
    dist_order = np.argsort(np.abs(obs_strikes - target))[:N_NEIGHBORS]
    if len(dist_order) < 3:
        return np.nan
    coeffs = np.polyfit(obs_strikes[dist_order], obs_ivs[dist_order], deg=2)
    if coeffs[0] > CONV_THRESH:          # convexity check
        return float(np.polyval(coeffs, target))
    return np.nan

def _linear_boundary_extrap(obs_strikes, obs_ivs, target):
    """2-point slope extrapolation from the nearest observed boundary."""
    if target < obs_strikes[0]:
        slope = (obs_ivs[1] - obs_ivs[0]) / (obs_strikes[1] - obs_strikes[0])
        return float(obs_ivs[0] + slope * (target - obs_strikes[0]))
    slope = (obs_ivs[-1] - obs_ivs[-2]) / (obs_strikes[-1] - obs_strikes[-2])
    return float(obs_ivs[-1] + slope * (target - obs_strikes[-1]))

# ----- LOO validation -----
loo_errors_ours   = []
loo_errors_linear = []

for idx in df.index:
    row = df.iloc[idx]
    for strikes, cols in [(ce_strikes, ce_cols), (pe_strikes, pe_cols)]:
        ivs = np.array([row[c] for c in cols], dtype=float)
        obs_mask = ~np.isnan(ivs)
        if obs_mask.sum() < 13:
            continue

        obs_idx  = np.where(obs_mask)[0]
        is_spike = float(np.nanmax(ivs[obs_mask])) > SPIKE_THRESH

        for leave_i in obs_idx:
            train_idx = obs_idx[obs_idx != leave_i]
            if len(train_idx) < 2:
                continue

            obs_s  = strikes[train_idx]
            obs_v  = ivs[train_idx]
            y_true = ivs[leave_i]
            x_tgt  = strikes[leave_i]

            lin_fn  = interp1d(obs_s, obs_v, kind='linear',
                               bounds_error=False, fill_value=np.nan)
            v_lin   = float(lin_fn(x_tgt))
            is_interior = np.isfinite(v_lin)
            if not is_interior:
                v_lin = _linear_boundary_extrap(obs_s, obs_v, x_tgt)

            loo_errors_linear.append((y_true - v_lin) ** 2)

            if not is_spike:
                q = _local_quad_estimate(obs_s, obs_v, x_tgt)
                if np.isfinite(q):
                    w = W_INTERIOR if is_interior else W_EDGE
                    pred = (1.0 - w) * v_lin + w * q
                else:
                    pred = v_lin
            else:
                pred = v_lin

            loo_errors_ours.append((y_true - pred) ** 2)

n_pts = len(loo_errors_ours)
print(f"LOO test points: {n_pts}")
print(f"Linear-only baseline MSE : {np.mean(loo_errors_linear):.8f}")
print(f"This model MSE           : {np.mean(loo_errors_ours):.8f}")
print(f"Improvement              : {(np.mean(loo_errors_linear)-np.mean(loo_errors_ours))/np.mean(loo_errors_linear)*100:.2f}%")


LOO test points: 4911
Linear-only baseline MSE : 0.00004980
This model MSE           : 0.00004848
Improvement              : 2.64%


In [4]:
def impute_wing(ivs: np.ndarray, strikes: np.ndarray) -> np.ndarray:
    """
    Imputes all missing values for one option type (CE or PE) at one timestamp.

    Strategy
    --------
    For each missing position:
      1. Attempt linear interpolation from all observed strikes.
         - If the target falls within the observed range  → interior gap (interpolation).
         - If outside                                     → edge gap (extrapolation).
      2. Attempt a local convex quadratic fit on the 5 nearest observed strikes.
         - Accept only if the leading coefficient is positive (convex smile shape).
      3. Blend:
         - Interior gap + valid quad : (1 - W_INTERIOR) * linear + W_INTERIOR * quad
         - Edge gap    + valid quad  : (1 - W_EDGE)     * linear + W_EDGE     * quad
         - No valid quad / spike regime : linear only
      4. Clip result to [IV_FLOOR, IV_CAP].

    No-Lookahead: uses only same-timestamp strikes — purely cross-sectional.
    """
    out      = ivs.copy()
    obs_mask = np.isfinite(out)

    if obs_mask.sum() < 2:
        return out   # defer to temporal fallback

    obs_s    = strikes[obs_mask]
    obs_v    = out[obs_mask]
    miss_idx = np.where(~obs_mask)[0]

    if len(miss_idx) == 0:
        return out

    is_spike = float(np.nanmax(obs_v)) > SPIKE_THRESH
    lin_fn   = interp1d(obs_s, obs_v, kind='linear',
                        bounds_error=False, fill_value=np.nan)

    for j in miss_idx:
        target = strikes[j]
        v_lin  = float(lin_fn(target))

        is_interior = np.isfinite(v_lin)
        if not is_interior:
            v_lin = _linear_boundary_extrap(obs_s, obs_v, target)

        if not is_spike:
            q = _local_quad_estimate(obs_s, obs_v, target)
            if np.isfinite(q):
                w     = W_INTERIOR if is_interior else W_EDGE
                final = (1.0 - w) * v_lin + w * q
            else:
                final = v_lin
        else:
            final = v_lin

        out[j] = float(np.clip(final, IV_FLOOR, IV_CAP))

    return out


# --- Main imputation loop ---
df_filled = df.copy()
all_iv_cols = ce_cols + pe_cols

for idx in df_filled.index:
    ce_vals = df_filled.loc[idx, ce_cols].values.astype(float)
    pe_vals = df_filled.loc[idx, pe_cols].values.astype(float)
    df_filled.loc[idx, ce_cols] = impute_wing(ce_vals, ce_strikes)
    df_filled.loc[idx, pe_cols] = impute_wing(pe_vals, pe_strikes)

n_remaining = df_filled[all_iv_cols].isna().sum().sum()
print(f"After cross-sectional pass — remaining missing: {n_remaining}")


After cross-sectional pass — remaining missing: 0


In [5]:
# --- Strictly causal temporal fallback ---
# Forward-fill only (past → present).
df_filled = df_filled.sort_values('datetime').reset_index(drop=True)
df_filled[all_iv_cols] = df_filled[all_iv_cols].ffill(axis=0)

# Safety net: for any row where entire wing is missing, use row mean
still_missing = df_filled[all_iv_cols].isna().any(axis=1)
if still_missing.any():
    row_means = df_filled.loc[still_missing, all_iv_cols].mean(axis=1)
    for col in all_iv_cols:
        df_filled.loc[still_missing, col] = df_filled.loc[still_missing, col].fillna(row_means)

df_filled[all_iv_cols] = df_filled[all_iv_cols].fillna(0.0)

print(f"Final missing count: {df_filled[all_iv_cols].isna().sum().sum()}")
print("Surface reconstruction complete.")


Final missing count: 0
Surface reconstruction complete.


In [6]:
df_filled.to_csv(FILLED_PATH, index=False)
print(f"Filled dataset saved → {FILLED_PATH}")


Filled dataset saved → ../data/filled_dataset.csv


In [7]:
SEPARATOR = "||"

def generate_solution(original_path: str, filled_path: str, output_path: str):
    """
    Convert the filled dataset into the required Kaggle submission format.
    Extract only the originally-missing entries and write them as (id, value) pairs.
    """
    original = pd.read_csv(original_path)
    filled   = pd.read_csv(filled_path)

    feat_cols = [c for c in original.columns if c != "datetime"]
    rows = []

    for col in feat_cols:
        was_missing = original[col].isna()
        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    solution.to_csv(output_path, index=False)
    print(f"Submission saved → {output_path}  ({len(solution)} rows)")
    return solution


sol = generate_solution(DATASET_PATH, FILLED_PATH, SUBMISSION_PATH)

print("\nSample predictions:")
print(sol.head(10).to_string(index=False))
print(f"\nValue range: {sol['value'].min():.5f} – {sol['value'].max():.5f}")


Submission saved → ../submissions/submission.csv  (5460 rows)

Sample predictions:
                                   id    value
07-01-2026 09:15||NIFTY27JAN2624100PE 0.163440
07-01-2026 09:15||NIFTY27JAN2625500CE 0.113876
07-01-2026 09:15||NIFTY27JAN2625800CE 0.101139
07-01-2026 09:20||NIFTY27JAN2624000PE 0.170055
07-01-2026 09:20||NIFTY27JAN2624200PE 0.159770
07-01-2026 09:20||NIFTY27JAN2624800PE 0.128225
07-01-2026 09:20||NIFTY27JAN2625000PE 0.118580
07-01-2026 09:20||NIFTY27JAN2625300CE 0.096810
07-01-2026 09:20||NIFTY27JAN2625400CE 0.107300
07-01-2026 09:20||NIFTY27JAN2625800CE 0.105666

Value range: 0.01831 – 5.79499
